In [2]:
import os, json, time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, Conv2D, MaxPool2D, Flatten,
                                     Dense, GlobalAveragePooling2D)


2026-08-21 20:41:51.898804: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787337711.963603  127395 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787337711.981459  127395 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-21 20:41:52.109604: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

DATA = "/home/admins/rebuild_workspace/06_dataset_final"
OUT  = "/home/admins/rebuild_workspace/models_comparison"
os.makedirs(OUT, exist_ok=True)

word_classes = ['bat','cup','drop','eat','fish','hot','jump','milk','pen','red']
IDG = ImageDataGenerator(rescale=1/255)

FILTERS = [32, 64, 128, 256, 512]

def make_generators():
    tr = IDG.flow_from_directory(os.path.join(DATA,"Train"), target_size=(224,224),
            classes=word_classes, batch_size=16, class_mode='categorical')
    va = IDG.flow_from_directory(os.path.join(DATA,"Validation"), target_size=(224,224),
            classes=word_classes, batch_size=16, class_mode='categorical', shuffle=False)
    return tr, va

train, valid = make_generators()
print(train.samples, valid.samples)

Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.
900 200


In [ ]:
def build(n_blocks, head):
    """head: 'flatten' or 'gap'"""
    layers = [Input(shape=(224,224,3))]
    for i in range(n_blocks):
        layers.append(Conv2D(FILTERS[i], (3,3), activation='relu', padding='same'))
        layers.append(MaxPool2D((2,2), strides=2))
    layers.append(Flatten() if head == 'flatten' else GlobalAveragePooling2D())
    layers += [Dense(256, activation='tanh'),
               Dense(128, activation='tanh'),
               Dense(10,  activation='softmax')]
    return Sequential(layers)

CONFIGS = {
    "c0_2blocks_flatten": (2, 'flatten'),
    "c1_5blocks_flatten": (5, 'flatten'),
    "c2_2blocks_gap":     (2, 'gap'),
    "c3_5blocks_gap":     (5, 'gap'),
}

for name, (nb, head) in CONFIGS.items():
    m = build(nb, head)
    print(f"{name:24s} params: {m.count_params():>12,}")
    del m

I0000 00:00:1787331204.388555   66494 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5658 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050, pci bus id: 0000:08:00.0, compute capability: 8.6


c0_2blocks_flatten       params:   51,434,058
c1_5blocks_flatten       params:    8,025,546
c2_2blocks_gap           params:       70,218
c3_5blocks_gap           params:    1,734,090


In [ ]:
SEEDS = [11, 22, 33]
EPOCHS = 15
results = []

for name, (nb, head) in CONFIGS.items():
    for seed in SEEDS:
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(seed)

        train, valid = make_generators()
        model = build(nb, head)
        model.compile(optimizer=Adam(learning_rate=0.0001),
                      loss='categorical_crossentropy', metrics=['accuracy'])

        t0 = time.time()
        hist = model.fit(train, validation_data=valid, epochs=EPOCHS, verbose=0)
        elapsed = time.time() - t0

        h = hist.history
        best_ep = int(np.argmin(h['val_loss']))
        row = dict(config=name, seed=seed, params=model.count_params(),
                   final_train_acc=h['accuracy'][-1],
                   final_val_acc=h['val_accuracy'][-1],
                   best_val_acc=max(h['val_accuracy']),
                   best_val_loss=min(h['val_loss']),
                   best_epoch=best_ep + 1,
                   seconds=round(elapsed, 1))
        results.append(row)
        # pd.DataFrame(h).to_csv(f"{OUT}/history_{name}_seed{seed}.csv", index=False)
        # model.save(f"{OUT}/{name}_seed{seed}.keras")

        print(f"{name:24s} seed {seed}  "
              f"train {row['final_train_acc']:.4f}  "
              f"val {row['final_val_acc']:.4f}  "
              f"best_val {row['best_val_acc']:.4f} @ep{row['best_epoch']}  "
              f"({row['seconds']}s)")

res = pd.DataFrame(results)
res.to_csv(f"{OUT}/architecture_comparison_log.csv", index=False)

Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.


/home/admins/miniconda3/envs/the_sheff/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
I0000 00:00:1787331358.421959   66603 service.cc:148] XLA service 0x7c8bd0005f90 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1787331358.421998   66603 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 3050, Compute Capability 8.6
2026-08-21 18:55:58.464587: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1787331358.587736   66603 cuda_dnn.cc:529] Loaded cuDNN version 92000
I0000 00:00:1787331363.654753   6

c0_2blocks_flatten       seed 11  train 1.0000  val 0.6750  best_val 0.7250 @ep11  (46.3s)
Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.


/home/admins/miniconda3/envs/the_sheff/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


c0_2blocks_flatten       seed 22  train 1.0000  val 0.7050  best_val 0.7250 @ep11  (37.2s)
Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.


...

c3_5blocks_gap           seed 11  train 0.4811  val 0.4150  best_val 0.4150 @ep15  (46.2s)
Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.


/home/admins/miniconda3/envs/the_sheff/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


c3_5blocks_gap           seed 22  train 0.4822  val 0.4100  best_val 0.4100 @ep15  (41.5s)
Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.


/home/admins/miniconda3/envs/the_sheff/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


c3_5blocks_gap           seed 33  train 0.4978  val 0.3450  best_val 0.3850 @ep14  (41.5s)


In [6]:
FILTERS = [32, 64, 128, 256, 512, 512]

def build2(n_blocks, bottleneck=None, extra_pool=False, dense1=256):
    layers = [Input(shape=(224,224,3))]
    for i in range(n_blocks):
        layers.append(Conv2D(FILTERS[i], (3,3), activation='relu', padding='same'))
        layers.append(MaxPool2D((2,2), strides=2))
    if bottleneck:
        layers.append(Conv2D(bottleneck, (1,1), activation='relu'))
    if extra_pool:
        layers.append(MaxPool2D((2,2), strides=2))
    layers.append(Flatten())
    layers += [Dense(dense1, activation='tanh'),
               Dense(128, activation='tanh'),
               Dense(10, activation='softmax')]
    return Sequential(layers)

CONFIGS2 = {
    "e0_5blocks":            dict(n_blocks=5),
    "e1_5blocks_1x1conv32":  dict(n_blocks=5, bottleneck=32),
    "e2_5blocks_extrapool":  dict(n_blocks=5, extra_pool=True),
    "e3_6blocks":            dict(n_blocks=6),
    "e4_4blocks":            dict(n_blocks=4),
}

for name, kw in CONFIGS2.items():
    m = build2(**kw)
    print(f"{name:24s} params: {m.count_params():>12,}")
    del m

e0_5blocks               params:    8,025,546
e1_5blocks_1x1conv32     params:    2,020,842
e2_5blocks_extrapool     params:    2,782,666
e3_6blocks               params:    5,142,474
e4_4blocks               params:   13,267,914


In [7]:
SEEDS2 = [11, 22, 33, 44, 55]
EPOCHS = 15
results2 = []

for name, kw in CONFIGS2.items():
    for seed in SEEDS2:
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(seed)

        train, valid = make_generators()
        model = build2(**kw)
        model.compile(optimizer=Adam(learning_rate=0.0001),
                      loss='categorical_crossentropy', metrics=['accuracy'])

        t0 = time.time()
        hist = model.fit(train, validation_data=valid, epochs=EPOCHS, verbose=0)
        elapsed = time.time() - t0

        h = hist.history
        row = dict(config=name, seed=seed, params=model.count_params(),
                   final_train_acc=h['accuracy'][-1],
                   final_val_acc=h['val_accuracy'][-1],
                   best_val_acc=max(h['val_accuracy']),
                   best_val_loss=min(h['val_loss']),
                   best_epoch=int(np.argmin(h['val_loss'])) + 1,
                   seconds=round(elapsed, 1))
        results2.append(row)
        pd.DataFrame(h).to_csv(f"{OUT}/history_{name}_seed{seed}.csv", index=False)

        print(f"{name:24s} seed {seed}  train {row['final_train_acc']:.4f}  "
              f"best_val {row['best_val_acc']:.4f} @ep{row['best_epoch']}  ({row['seconds']}s)")

res2 = pd.DataFrame(results2)
res2.to_csv(f"{OUT}/architecture_comparison_round2_log.csv", index=False)

print("\n--- summary ---")
print(res2.groupby('config').agg(
    params=('params','first'),
    mean_best_val=('best_val_acc','mean'),
    std_best_val=('best_val_acc','std'),
    min_val=('best_val_acc','min'),
    max_val=('best_val_acc','max'),
    mean_train=('final_train_acc','mean')).round(4).to_string())

Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.


/home/admins/miniconda3/envs/the_sheff/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


e0_5blocks               seed 11  train 0.9944  best_val 0.7200 @ep10  (44.1s)
Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.


/home/admins/miniconda3/envs/the_sheff/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


e0_5blocks               seed 22  train 0.9878  best_val 0.7350 @ep15  (42.3s)
Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.


...

e4_4blocks               seed 33  train 0.9989  best_val 0.6800 @ep11  (39.0s)
Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.


/home/admins/miniconda3/envs/the_sheff/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


e4_4blocks               seed 44  train 1.0000  best_val 0.6750 @ep9  (38.9s)
Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.


/home/admins/miniconda3/envs/the_sheff/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


e4_4blocks               seed 55  train 1.0000  best_val 0.7000 @ep11  (38.9s)

--- summary ---
                        params  mean_best_val  std_best_val  min_val  max_val  mean_train
config                                                                                   
e0_5blocks             8025546          0.733        0.0342    0.700    0.790      0.9913
e1_5blocks_1x1conv32   2020842          0.206        0.2204    0.100    0.600      0.2400
e2_5blocks_extrapool   2782666          0.743        0.0241    0.725    0.785      0.9882
e3_6blocks             5142474          0.728        0.0228    0.705    0.755      0.9724
e4_4blocks            13267914          0.688        0.0115    0.675    0.700      0.9998


In [5]:
def build3(n_blocks, extra_pool=False, act='tanh'):
    layers = [Input(shape=(224,224,3))]
    for i in range(n_blocks):
        layers.append(Conv2D(FILTERS[i], (3,3), activation='relu', padding='same'))
        layers.append(MaxPool2D((2,2), strides=2))
    if extra_pool:
        layers.append(MaxPool2D((2,2), strides=2))
    layers.append(Flatten())
    layers += [Dense(256, activation=act),
               Dense(128, activation=act),
               Dense(10, activation='softmax')]
    return Sequential(layers)

CONFIGS3 = {
    "f0_e2_tanh": dict(n_blocks=5, extra_pool=True, act='tanh'),
    "f1_e2_relu": dict(n_blocks=5, extra_pool=True, act='relu'),
    "f2_c0_relu": dict(n_blocks=2, extra_pool=False, act='relu'),
}

for name, kw in CONFIGS3.items():
    m = build3(**kw)
    print(f"{name:16s} params: {m.count_params():>12,}")
    del m

I0000 00:00:1787337742.623831  127395 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5658 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050, pci bus id: 0000:08:00.0, compute capability: 8.6


f0_e2_tanh       params:    2,782,666
f1_e2_relu       params:    2,782,666
f2_c0_relu       params:   51,434,058


In [6]:
SEEDS3 = [11, 22, 33, 44, 55]
EPOCHS = 15
results3 = []

for name, kw in CONFIGS3.items():
    for seed in SEEDS3:
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(seed)

        train, valid = make_generators()
        model = build3(**kw)
        model.compile(optimizer=Adam(learning_rate=0.0001),
                      loss='categorical_crossentropy', metrics=['accuracy'])

        t0 = time.time()
        hist = model.fit(train, validation_data=valid, epochs=EPOCHS, verbose=0)
        elapsed = time.time() - t0

        h = hist.history
        row = dict(config=name, seed=seed, params=model.count_params(),
                   final_train_acc=h['accuracy'][-1],
                   final_val_acc=h['val_accuracy'][-1],
                   best_val_acc=max(h['val_accuracy']),
                   best_val_loss=min(h['val_loss']),
                   best_epoch=int(np.argmin(h['val_loss'])) + 1,
                   seconds=round(elapsed, 1))
        results3.append(row)
        pd.DataFrame(h).to_csv(f"{OUT}/history_{name}_seed{seed}.csv", index=False)

        print(f"{name:24s} seed {seed}  train {row['final_train_acc']:.4f}  "
              f"best_val {row['best_val_acc']:.4f} @ep{row['best_epoch']}  ({row['seconds']}s)")

res3 = pd.DataFrame(results3)
res3.to_csv(f"{OUT}/architecture_comparison_round3_log.csv", index=False)

print("\n--- summary ---")
print(res3.groupby('config').agg(
    params=('params','first'),
    mean_best_val=('best_val_acc','mean'),
    std_best_val=('best_val_acc','std'),
    min_val=('best_val_acc','min'),
    max_val=('best_val_acc','max'),
    mean_train=('final_train_acc','mean')).round(4).to_string())

Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.


/home/admins/miniconda3/envs/the_sheff/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
I0000 00:00:1787336372.802232  111192 service.cc:148] XLA service 0x7b87a0005f10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1787336372.802269  111192 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 3050, Compute Capability 8.6
2026-08-21 20:19:32.839468: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1787336373.027682  111192 cuda_dnn.cc:529] Loaded cuDNN version 92000
I0000 00:00:1787336380.339844  11

f0_e2_tanh               seed 11  train 0.9878  best_val 0.7600 @ep11  (53.5s)
Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.


/home/admins/miniconda3/envs/the_sheff/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


f0_e2_tanh               seed 22  train 0.9833  best_val 0.7300 @ep12  (43.7s)
Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.


...

f2_c0_relu               seed 33  train 1.0000  best_val 0.6800 @ep11  (42.3s)
Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.


/home/admins/miniconda3/envs/the_sheff/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


f2_c0_relu               seed 44  train 0.9989  best_val 0.6700 @ep8  (40.0s)
Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.


/home/admins/miniconda3/envs/the_sheff/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


f2_c0_relu               seed 55  train 0.9989  best_val 0.7350 @ep11  (37.2s)

--- summary ---
              params  mean_best_val  std_best_val  min_val  max_val  mean_train
config                                                                         
f0_e2_tanh   2782666          0.745        0.0255    0.715    0.780      0.9862
f1_e2_relu   2782666          0.660        0.0310    0.625    0.695      0.9140
f2_c0_relu  51434058          0.694        0.0338    0.660    0.735      0.9967


In [ ]:
CHOSEN = dict(n_blocks=5, extra_pool=True, act='tanh')
FINAL_SEEDS = [11, 22, 33]

test = IDG.flow_from_directory(os.path.join(DATA,"Test"), target_size=(224,224),
        classes=word_classes, batch_size=16, class_mode='categorical', shuffle=False)

REAL_DIR = "/home/admins/rebuild_workspace/06_test_real_only"
test_real = IDG.flow_from_directory(REAL_DIR, target_size=(224,224),
        classes=word_classes, batch_size=16, class_mode='categorical', shuffle=False)

final_rows = []
for seed in FINAL_SEEDS:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)

    train, valid = make_generators()
    model = build3(**CHOSEN)
    model.compile(optimizer=Adam(learning_rate=0.0001),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    hist = model.fit(train, validation_data=valid, epochs=15, verbose=0)

    lf, af = model.evaluate(test, verbose=0)
    lr, ar = model.evaluate(test_real, verbose=0)

    final_rows.append(dict(seed=seed,
                           val_acc=hist.history['val_accuracy'][-1],
                           test_full_acc=af, test_full_loss=lf,
                           test_real_acc=ar, test_real_loss=lr))
#     model.save(f"{OUT}/final_5blocks_extrapool_seed{seed}.keras")
    pd.DataFrame(hist.history).to_csv(f"{OUT}/history_final_seed{seed}.csv", index=False)
    print(f"seed {seed}  val {hist.history['val_accuracy'][-1]:.4f}  "
          f"test_full {af:.4f}  test_real {ar:.4f}")

fin = pd.DataFrame(final_rows)
fin.to_csv(f"{OUT}/final_model_evaluation_log.csv", index=False)

print("\n--- vs baseline ---")
print(f"                    baseline    new (mean of 3)")
print(f"full test (200)      0.7150      {fin.test_full_acc.mean():.4f}  "
      f"(range {fin.test_full_acc.min():.4f}-{fin.test_full_acc.max():.4f})")
print(f"real only (90)       0.7556      {fin.test_real_acc.mean():.4f}  "
      f"(range {fin.test_real_acc.min():.4f}-{fin.test_real_acc.max():.4f})")

Found 200 images belonging to 10 classes.


Found 90 images belonging to 10 classes.
Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.


/home/admins/miniconda3/envs/the_sheff/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
I0000 00:00:1787337765.187979  128842 service.cc:148] XLA service 0x7b95680100f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1787337765.188018  128842 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 3050, Compute Capability 8.6
2026-08-21 20:42:45.251088: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1787337765.467570  128842 cuda_dnn.cc:529] Loaded cuDNN version 92000
I0000 00:00:1787337772.565635  12

seed 11  val 0.7200  test_full 0.6700  test_real 0.7222
Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.


/home/admins/miniconda3/envs/the_sheff/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


seed 22  val 0.7450  test_full 0.6750  test_real 0.7000
Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.


/home/admins/miniconda3/envs/the_sheff/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


seed 33  val 0.7600  test_full 0.6800  test_real 0.7222

--- vs baseline ---
                    baseline    new (mean of 3)
full test (200)      0.7150      0.6750  (range 0.6700-0.6800)
real only (90)       0.7556      0.7148  (range 0.7000-0.7222)
